In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Find project root
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"

master_path = PROCESSED / "master_panel.parquet"

print("Master panel exists?", master_path.exists())

df = pd.read_parquet(master_path)

print("Shape:", df.shape)
print(df.columns.tolist())
df.head()

Master panel exists? True
Shape: (479783, 27)
['permno', 'permco', 'ticker', 'cusip', 'issuernm', 'siccd', 'naics', 'dlyclose', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyprc', 'dlyret', 'dlyretx', 'dlyvol', 'shrout', 'dlycap', 'sprtrn', 'vwretd', 'ewretd', 'date', 'quarter', 'numeric_transparency', 'analyst_selectivity_ratio', 'language_complexity', 'net_positivity', 'quarter_end']


,permno,permco,ticker,cusip,issuernm,siccd,naics,dlyclose,dlyopen,dlyhigh,...,sprtrn,vwretd,ewretd,date,quarter,numeric_transparency,analyst_selectivity_ratio,language_complexity,net_positivity,quarter_end
0,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.21,57.10,57.100,...,-0.008862,-0.008757,-0.004051,2014-01-02,None,NaN,NaN,NaN,NaN,NaT
1,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.92,56.39,57.345,...,-0.000333,0.000491,0.004096,2014-01-03,None,NaN,NaN,NaN,NaN,NaT
2,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.64,57.40,57.700,...,-0.002512,-0.003340,-0.001676,2014-01-06,None,NaN,NaN,NaN,NaN,NaT
3,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,57.45,56.95,57.630,...,0.006082,0.006090,0.006892,2014-01-07,None,NaN,NaN,NaN,NaN,NaT
4,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,58.39,57.33,58.540,...,-0.000212,0.000155,0.000835,2014-01-08,None,NaN,NaN,NaN,NaN,NaT


In [3]:
# Standardize ticker and dates again just to be safe
df["ticker"] = df["ticker"].astype(str).str.upper().str.strip()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["quarter_end"] = pd.to_datetime(df["quarter_end"], errors="coerce")

# Sort for rolling/group operations
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

# Keep only rows with valid ticker/date
df = df.dropna(subset=["ticker", "date"]).copy()

print(df.shape)
df[["ticker", "date"]].head()

(479783, 27)


,ticker,date
0,A,2014-01-02
1,A,2014-01-03
2,A,2014-01-06
3,A,2014-01-07
4,A,2014-01-08


In [4]:
# Return-based momentum features
df["ret_5d"] = df.groupby("ticker")["dlyclose"].transform(lambda s: s / s.shift(5) - 1)
df["ret_20d"] = df.groupby("ticker")["dlyclose"].transform(lambda s: s / s.shift(20) - 1)
df["ret_60d"] = df.groupby("ticker")["dlyclose"].transform(lambda s: s / s.shift(60) - 1)

df[["ticker", "date", "dlyclose", "ret_5d", "ret_20d", "ret_60d"]].head(10)

,ticker,date,dlyclose,ret_5d,ret_20d,ret_60d
0,A,2014-01-02,56.21,NaN,NaN,NaN
1,A,2014-01-03,56.92,NaN,NaN,NaN
2,A,2014-01-06,56.64,NaN,NaN,NaN
3,A,2014-01-07,57.45,NaN,NaN,NaN
4,A,2014-01-08,58.39,NaN,NaN,NaN
5,A,2014-01-09,58.41,0.039139,NaN,NaN
6,A,2014-01-10,58.93,0.035313,NaN,NaN
7,A,2014-01-13,58.93,0.040431,NaN,NaN
8,A,2014-01-14,59.88,0.042298,NaN,NaN
9,A,2014-01-15,60.34,0.033396,NaN,NaN


In [5]:
# Rolling volatility using daily return
df["vol_20d"] = df.groupby("ticker")["dlyret"].transform(lambda s: s.rolling(20).std())
df["vol_60d"] = df.groupby("ticker")["dlyret"].transform(lambda s: s.rolling(60).std())

df[["ticker", "date", "dlyret", "vol_20d", "vol_60d"]].head(10)

,ticker,date,dlyret,vol_20d,vol_60d
0,A,2014-01-02,-0.017136,NaN,NaN
1,A,2014-01-03,0.012631,NaN,NaN
2,A,2014-01-06,-0.004919,NaN,NaN
3,A,2014-01-07,0.014301,NaN,NaN
4,A,2014-01-08,0.016362,NaN,NaN
5,A,2014-01-09,0.000343,NaN,NaN
6,A,2014-01-10,0.008903,NaN,NaN
7,A,2014-01-13,0.000000,NaN,NaN
8,A,2014-01-14,0.016121,NaN,NaN
9,A,2014-01-15,0.007682,NaN,NaN


In [6]:
# Volume features
df["avg_volume_20d"] = df.groupby("ticker")["dlyvol"].transform(lambda s: s.rolling(20).mean())
df["relative_volume"] = df["dlyvol"] / df["avg_volume_20d"]

df[["ticker", "date", "dlyvol", "avg_volume_20d", "relative_volume"]].head(10)

,ticker,date,dlyvol,avg_volume_20d,relative_volume
0,A,2014-01-02,1916200.0,NaN,NaN
1,A,2014-01-03,1866700.0,NaN,NaN
2,A,2014-01-06,1777500.0,NaN,NaN
3,A,2014-01-07,1463200.0,NaN,NaN
4,A,2014-01-08,2659500.0,NaN,NaN
5,A,2014-01-09,1757600.0,NaN,NaN
6,A,2014-01-10,1623300.0,NaN,NaN
7,A,2014-01-13,2946700.0,NaN,NaN
8,A,2014-01-14,2562200.0,NaN,NaN
9,A,2014-01-15,2335200.0,NaN,NaN


In [8]:
# First create a quarter-level view for transcript deltas
quarter_df = (
    df[["ticker", "quarter", "quarter_end",
        "net_positivity", "numeric_transparency",
        "language_complexity", "analyst_selectivity_ratio"]]
    .drop_duplicates(subset=["ticker", "quarter_end"])
    .sort_values(["ticker", "quarter_end"])
    .copy()
)

quarter_df["positivity_delta"] = quarter_df.groupby("ticker")["net_positivity"].diff()
quarter_df["transparency_delta"] = quarter_df.groupby("ticker")["numeric_transparency"].diff()
quarter_df["complexity_delta"] = quarter_df.groupby("ticker")["language_complexity"].diff()
quarter_df["selectivity_delta"] = quarter_df.groupby("ticker")["analyst_selectivity_ratio"].diff()

quarter_df.head()

,ticker,quarter,quarter_end,net_positivity,numeric_transparency,language_complexity,analyst_selectivity_ratio,positivity_delta,transparency_delta,complexity_delta,selectivity_delta
60,A,CQ12014,2014-03-31,0.25,2.67,11.32,64.71,NaN,NaN,NaN,NaN
123,A,CQ22014,2014-06-30,0.70,1.90,11.64,50.00,0.45,-0.77,0.32,-14.71
187,A,CQ32014,2014-09-30,0.97,2.19,11.91,78.57,0.27,0.29,0.27,28.57
251,A,CQ42014,2014-12-31,0.75,3.05,12.15,64.29,-0.22,0.86,0.24,-14.28
312,A,CQ12015,2015-03-31,0.93,1.96,12.42,75.00,0.18,-1.09,0.27,10.71


In [9]:
# Merge quarter-level delta features back into daily dataset
delta_cols = [
    "ticker", "quarter_end",
    "positivity_delta",
    "transparency_delta",
    "complexity_delta",
    "selectivity_delta"
]

df = df.merge(
    quarter_df[delta_cols],
    on=["ticker", "quarter_end"],
    how="left"
)

df[[
    "ticker", "date", "quarter_end",
    "positivity_delta", "transparency_delta",
    "complexity_delta", "selectivity_delta"
]].head(10)

,ticker,date,quarter_end,positivity_delta,transparency_delta,complexity_delta,selectivity_delta
0,A,2014-01-02,NaT,NaN,NaN,NaN,NaN
1,A,2014-01-03,NaT,NaN,NaN,NaN,NaN
2,A,2014-01-06,NaT,NaN,NaN,NaN,NaN
3,A,2014-01-07,NaT,NaN,NaN,NaN,NaN
4,A,2014-01-08,NaT,NaN,NaN,NaN,NaN
5,A,2014-01-09,NaT,NaN,NaN,NaN,NaN
6,A,2014-01-10,NaT,NaN,NaN,NaN,NaN
7,A,2014-01-13,NaT,NaN,NaN,NaN,NaN
8,A,2014-01-14,NaT,NaN,NaN,NaN,NaN
9,A,2014-01-15,NaT,NaN,NaN,NaN,NaN


In [10]:
# Fill key features temporarily for score construction
df["net_positivity_filled"] = df["net_positivity"].fillna(0)
df["numeric_transparency_filled"] = df["numeric_transparency"].fillna(0)
df["language_complexity_filled"] = df["language_complexity"].fillna(0)
df["ret_20d_filled"] = df["ret_20d"].fillna(0)
df["vol_20d_filled"] = df["vol_20d"].fillna(0)

# 1. Credibility score
df["credibility_score"] = (
    0.4 * df["net_positivity_filled"] +
    0.4 * df["numeric_transparency_filled"] -
    0.2 * df["language_complexity_filled"]
)

# 2. Risk score
df["risk_score"] = (
    df["vol_20d_filled"] +
    0.5 * df["language_complexity_filled"] -
    0.5 * df["ret_20d_filled"]
)

# 3. Misalignment score
df["misalignment_score"] = (
    df["net_positivity_filled"] - df["ret_20d_filled"]
)

df[[
    "ticker", "date",
    "credibility_score", "risk_score", "misalignment_score"
]].head(10)

,ticker,date,credibility_score,risk_score,misalignment_score
0,A,2014-01-02,0.0,0.0,0.0
1,A,2014-01-03,0.0,0.0,0.0
2,A,2014-01-06,0.0,0.0,0.0
3,A,2014-01-07,0.0,0.0,0.0
4,A,2014-01-08,0.0,0.0,0.0
5,A,2014-01-09,0.0,0.0,0.0
6,A,2014-01-10,0.0,0.0,0.0
7,A,2014-01-13,0.0,0.0,0.0
8,A,2014-01-14,0.0,0.0,0.0
9,A,2014-01-15,0.0,0.0,0.0


In [11]:
helper_cols = [
    "net_positivity_filled",
    "numeric_transparency_filled",
    "language_complexity_filled",
    "ret_20d_filled",
    "vol_20d_filled"
]

df = df.drop(columns=helper_cols, errors="ignore")

In [12]:
print("Final shape:", df.shape)

check_cols = [
    "ticker", "date",
    "ret_5d", "ret_20d", "ret_60d",
    "vol_20d", "vol_60d",
    "relative_volume",
    "positivity_delta", "transparency_delta",
    "complexity_delta", "selectivity_delta",
    "credibility_score", "risk_score", "misalignment_score"
]

available_check_cols = [c for c in check_cols if c in df.columns]

print(df[available_check_cols].head(10))
print("\nMissing values in new columns:")
print(df[available_check_cols].isna().sum())

Final shape: (479783, 45)
  ticker       date    ret_5d  ret_20d  ret_60d  vol_20d  vol_60d  \
0      A 2014-01-02       NaN      NaN      NaN      NaN      NaN   
1      A 2014-01-03       NaN      NaN      NaN      NaN      NaN   
2      A 2014-01-06       NaN      NaN      NaN      NaN      NaN   
3      A 2014-01-07       NaN      NaN      NaN      NaN      NaN   
4      A 2014-01-08       NaN      NaN      NaN      NaN      NaN   
5      A 2014-01-09  0.039139      NaN      NaN      NaN      NaN   
6      A 2014-01-10  0.035313      NaN      NaN      NaN      NaN   
7      A 2014-01-13  0.040431      NaN      NaN      NaN      NaN   
8      A 2014-01-14  0.042298      NaN      NaN      NaN      NaN   
9      A 2014-01-15  0.033396      NaN      NaN      NaN      NaN   

   relative_volume  positivity_delta  transparency_delta  complexity_delta  \
0              NaN               NaN                 NaN               NaN   
1              NaN               NaN                 NaN  

In [13]:
output_path = PROCESSED / "master_panel_features.parquet"
df.to_parquet(output_path, index=False)

print("Saved feature dataset to:")
print(output_path)

Saved feature dataset to:
c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\master_panel_features.parquet


In [14]:
test_df = pd.read_parquet(output_path)
print(test_df.shape)
test_df.head()

(479783, 45)


,permno,permco,ticker,cusip,issuernm,siccd,naics,dlyclose,dlyopen,dlyhigh,...,numeric_transparency_delta,language_complexity_delta,analyst_selectivity_ratio_delta,positivity_delta,transparency_delta,complexity_delta,selectivity_delta,credibility_score,risk_score,misalignment_score
0,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.21,57.10,57.100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
1,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.92,56.39,57.345,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
2,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,56.64,57.40,57.700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
3,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,57.45,56.95,57.630,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0
4,87432,36364,A,00846U10,AGILENT TECHNOLOGIES INC,3825,334515,58.39,57.33,58.540,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0


In [15]:
# Drop duplicate delta columns
drop_cols = [
    "net_positivity_delta",
    "numeric_transparency_delta",
    "language_complexity_delta",
    "analyst_selectivity_ratio_delta",
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

# Optional: make scores NaN when transcript data is missing
missing_transcript_mask = df[
    ["net_positivity", "numeric_transparency", "language_complexity", "analyst_selectivity_ratio"]
].isna().all(axis=1)

df.loc[missing_transcript_mask, ["credibility_score", "misalignment_score"]] = np.nan

# For risk_score, keep it if you want it partly market-based,
# or also null it out if you want transcript-dependent interpretation:
# df.loc[missing_transcript_mask, ["risk_score"]] = np.nan

# Save cleaned version again
output_path = PROCESSED / "master_panel_features.parquet"
df.to_parquet(output_path, index=False)

print("Saved cleaned feature dataset to:")
print(output_path)
print("New shape:", df.shape)
print(df.columns.tolist())

Saved cleaned feature dataset to:
c:\Users\mkp22\OneDrive\Desktop\FA Project\FA-Project\data\processed\master_panel_features.parquet
New shape: (479783, 41)
['permno', 'permco', 'ticker', 'cusip', 'issuernm', 'siccd', 'naics', 'dlyclose', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyprc', 'dlyret', 'dlyretx', 'dlyvol', 'shrout', 'dlycap', 'sprtrn', 'vwretd', 'ewretd', 'date', 'quarter', 'numeric_transparency', 'analyst_selectivity_ratio', 'language_complexity', 'net_positivity', 'quarter_end', 'ret_5d', 'ret_20d', 'ret_60d', 'vol_20d', 'vol_60d', 'avg_volume_20d', 'relative_volume', 'positivity_delta', 'transparency_delta', 'complexity_delta', 'selectivity_delta', 'credibility_score', 'risk_score', 'misalignment_score']
